In [1]:
import tensorflow as tf

model = tf.keras.models.load_model('lost_dogs_model2')

# view model architecture to confirm we have save and loaded correctly
model.summary()

ValueError: File format not supported: filepath=lost_dogs_model2. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(lost_dogs_model2, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def prep_data(text):
    tokens = tokenizer.encode_plus(text, max_length=512,
                                   truncation=True, padding='max_length',
                                   add_special_tokens=True, return_token_type_ids=False,
                                   return_tensors='tf')
    # tokenizer returns int32 tensors, we need to return float64, so we use tf.cast
    return {'input_ids': tf.cast(tokens['input_ids'], tf.float64),
            'attention_mask': tf.cast(tokens['attention_mask'], tf.float64)}

C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
probs = model.predict(prep_data("Gente perdí a mi perro, es negrito de tamaño mediano, ayudenlo a volver a casa"))[0]

probs

1/1 [==============================] - 1s 1s/step


array([0.72094995, 0.27905008], dtype=float32)

In [1]:
probs = model.predict(prep_data("Perro encontrado, parece desorientado, fue visto en La Molina por la avenida constructores info al 9945754"))[0]

probs

NameError: name 'model' is not defined

In [7]:
import pandas as pd

# so we can see full phrase
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('datos_sin procesar/data_preproc_todo.csv', sep=';')
df=df.drop(columns=['Unnamed: 0','tipoPost'])
df.head()

,text
0,perro perdido puerto real cabo rojo. !!!! ayudenme a encontrar al dueño
1,"#encontrado perro en la 9 con 50, es un perro muy calmado y estan buscando sus dueños. info al 3155335922. ayudaños a difundir para encontrar a sus dueños o alguien que desee adoptarlo."
2,"ayudenme a compartir para ver si alguien reconoce a este perro que parece perdido. trae collar sin placa, muy amigable, por metro puebla. @mascotassismo"
3,"difundir por favor perro macho encontrado en parque avellaneda, toda la descripcion esta en la foto lo tengo en mi casa momentaneamente pero necesito que me ayuden difundiendo, parece tener dueños! #perroperdido #perdido #difundir #parqueavellaneda"
4,"perro perdido este peludito fue recogido sin microchip esta mañana en la carretera cambre - o temple, a la altura de la urbanizacion a barcala, en el @concellocambre. se encuentra a nuestro cuidado en las instalaciones de liminon (#abegondo) 981 67 90 83"


In [8]:
import numpy as np
df['tipoPost'] = None

for i, row in df.iterrows():
    # get token tensors
    tokens = prep_data(row['text'])
    # get probabilities
    probs = model.predict(tokens)
    # find argmax for winning class
    pred = np.argmax(probs)
    # add to dataframe
    df.at[i, 'tipoPost'] = pred

df.head()

#1:Encontró
#0: Busca

1/1 [==============================] - 1s 1s/step


,text,tipoPost
0,perro perdido puerto real cabo rojo. !!!! ayudenme a encontrar al dueño,0
1,"#encontrado perro en la 9 con 50, es un perro muy calmado y estan buscando sus dueños. info al 3155335922. ayudaños a difundir para encontrar a sus dueños o alguien que desee adoptarlo.",1
2,"ayudenme a compartir para ver si alguien reconoce a este perro que parece perdido. trae collar sin placa, muy amigable, por metro puebla. @mascotassismo",1
3,"difundir por favor perro macho encontrado en parque avellaneda, toda la descripcion esta en la foto lo tengo en mi casa momentaneamente pero necesito que me ayuden difundiendo, parece tener dueños! #perroperdido #perdido #difundir #parqueavellaneda",0
4,"perro perdido este peludito fue recogido sin microchip esta mañana en la carretera cambre - o temple, a la altura de la urbanizacion a barcala, en el @concellocambre. se encuentra a nuestro cuidado en las instalaciones de liminon (#abegondo) 981 67 90 83",1


In [9]:
df

,text,tipoPost
0,perro perdido puerto real cabo rojo. !!!! ayudenme a encontrar al dueño,0
1,"#encontrado perro en la 9 con 50, es un perro muy calmado y estan buscando sus dueños. info al 3155335922. ayudaños a difundir para encontrar a sus dueños o alguien que desee adoptarlo.",1
2,"ayudenme a compartir para ver si alguien reconoce a este perro que parece perdido. trae collar sin placa, muy amigable, por metro puebla. @mascotassismo",1
3,"difundir por favor perro macho encontrado en parque avellaneda, toda la descripcion esta en la foto lo tengo en mi casa momentaneamente pero necesito que me ayuden difundiendo, parece tener dueños! #perroperdido #perdido #difundir #parqueavellaneda",0
4,"perro perdido este peludito fue recogido sin microchip esta mañana en la carretera cambre - o temple, a la altura de la urbanizacion a barcala, en el @concellocambre. se encuentra a nuestro cuidado en las instalaciones de liminon (#abegondo) 981 67 90 83",1
...,...,...
178,"comparte!! mascota perdida!!""spike"" se perdio por el sector del parque industrial #ambato.es un perrito snauzer, color pimienta, tamaño pequeno, requiere de cuidados especiales y medicamentos debido a su avanzada edad.contacto: 032434168, 0997179340, 0987728646, 0994851768.",1
179,nos ayudan por favor con miles de rtperrito llamado sparky se perdio hoy 15 de septiembre en la colonia del valle.se ofrece recompensa. informes conmigo.@mascotassismo @bjalcaldia #cdmx @en_ladelvalle @gobiernomx @gobcdmx @zazilcarreras @rickyklehr @_elan_ @yosoypambo,0
180,ayudaaaa porfaaasss @alcaldiagye parece perrito de casa que se escapo o lo dejaron botado. porfas ayudenlo! @josuesanchezec @cynthiaviteri6,0
181,perrito perdido por la cdla naval norte. parece que se escapo de su casa!! rt para que encuentre a su familia.,0
